# Output Guardrails

This notebook demonstrates how to implement **output guardrails** using the Strands Agents SDK's `AfterInvocationEvent` hook.

Output guardrails inspect model responses **after** inference completes, allowing you to:
- **BLOCK**: Replace the entire response with a safe fallback message
- **REDACT**: Replace only matched patterns (e.g., PII) with `[REDACTED]`

**Key concept:** Use a `HookProvider` class with `AfterInvocationEvent` to intercept responses after model inference. Access the agent's messages via `event.agent.messages` and modify the last assistant message.

## Setup

In [ ]:
# Install required packages
!pip install strands-agents strands-agents-tools --upgrade -q

In [ ]:
import logging
from strands.hooks import HookProvider, HookRegistry, AfterInvocationEvent

# Import content filters from our shared module
import content_filters as _filters_module
KeywordContentFilter = _filters_module.KeywordContentFilter
RegexContentFilter = _filters_module.RegexContentFilter
FormatComplianceFilter = _filters_module.FormatComplianceFilter
Severity = _filters_module.Severity
FilterResult = _filters_module.FilterResult
run_filters = _filters_module.run_filters

logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S")

## Helper: Modify the Last Assistant Message

Output guardrails work by modifying the conversation history. These helpers replace or redact the last assistant message content.

In [ ]:
def _replace_assistant_response(messages: list, new_text: str) -> None:
    """Replace the last assistant message content with new text."""
    if not messages:
        return
    for message in reversed(messages):
        if message.get("role") == "assistant":
            message["content"] = [{"text": new_text}]
            return


def _redact_assistant_response(messages: list, redacted_text: str) -> None:
    """Redact specific patterns in the last assistant message.
    
    Unlike full replacement, redaction preserves the overall response structure
    but replaces sensitive patterns with [REDACTED].
    """
    if not messages:
        return
    for message in reversed(messages):
        if message.get("role") == "assistant":
            new_content = []
            for block in message.get("content", []):
                if "text" in block:
                    new_content.append({"text": redacted_text})
                else:
                    new_content.append(block)
            message["content"] = new_content
            return

## Output Guardrail: BLOCK Behavior

The BLOCK behavior replaces the **entire** response with a safe fallback message when prohibited content is detected in the model's output.

In [ ]:
# Define prohibited keywords that should never appear in model output
OUTPUT_PROHIBITED_KEYWORDS = [
    "confidential internal",
    "classified information",
    "trade secret",
    "proprietary algorithm",
]

output_keyword_filter = KeywordContentFilter(
    name="output_topic_blocker",
    keywords=OUTPUT_PROHIBITED_KEYWORDS,
    severity=Severity.BLOCK,
)

BLOCKED_RESPONSE_FALLBACK = (
    "I'm sorry, but I cannot provide that information. "
    "The response was blocked by a content safety filter."
)


def output_guardrail_logic(messages: list) -> None:
    """Output guardrail that blocks responses containing prohibited content."""
    if not messages:
        return

    # Find the last assistant message
    last_assistant_text = None
    for message in reversed(messages):
        if message.get("role") == "assistant":
            for block in message.get("content", []):
                if "text" in block:
                    last_assistant_text = block["text"]
                    break
            break

    if not last_assistant_text:
        return

    result = output_keyword_filter.evaluate(last_assistant_text)

    if not result.passed:
        logger.warning(
            f"[OUTPUT GUARDRAIL] Blocked response. "
            f"Filter: {result.filter_name}, Reason: {result.message}"
        )
        _replace_assistant_response(messages, BLOCKED_RESPONSE_FALLBACK)
    else:
        logger.debug("[OUTPUT GUARDRAIL] Response passed keyword filter.")

## Output Guardrail: REDACT Behavior

The REDACT behavior replaces only the **matched patterns** (like PII) with `[REDACTED]`, preserving the rest of the response. This is less disruptive than blocking the entire response.

In [ ]:
# Define PII patterns for output redaction
OUTPUT_PII_PATTERNS = [
    r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b",  # Email
    r"\b\d{3}[-.]?\d{3}[-.]?\d{4}\b",  # Phone number
    r"\b\d{3}-\d{2}-\d{4}\b",  # SSN
]

pii_redaction_filter = RegexContentFilter(
    name="output_pii_redactor",
    patterns=OUTPUT_PII_PATTERNS,
    severity=Severity.REDACT,
)


def pii_output_guardrail_logic(messages: list) -> None:
    """Output guardrail that redacts PII from model responses."""
    if not messages:
        return

    # Find the last assistant message text
    last_assistant_text = None
    for message in reversed(messages):
        if message.get("role") == "assistant":
            for block in message.get("content", []):
                if "text" in block:
                    last_assistant_text = block["text"]
                    break
            break

    if not last_assistant_text:
        return

    result = pii_redaction_filter.evaluate(last_assistant_text)

    if not result.passed:
        logger.warning(
            f"[PII OUTPUT GUARDRAIL] Redacted PII from response. "
            f"Filter: {result.filter_name}, Reason: {result.message}"
        )
        _redact_assistant_response(messages, result.redacted_text)
    else:
        logger.debug("[PII OUTPUT GUARDRAIL] Response passed PII filter.")

## Combined Output Guardrail: Mixed Behaviors

Compose multiple output filters with different behaviors:
1. **Keyword filter (BLOCK)** — replaces entire response if triggered
2. **Format compliance filter (BLOCK)** — blocks code execution instructions
3. **PII filter (REDACT)** — redacts matched patterns only

BLOCK-severity violations are checked first. If none are found, REDACT filters clean up the response.

In [ ]:
format_filter = FormatComplianceFilter(
    name="output_format_compliance",
    severity=Severity.BLOCK,
)


def combined_output_guardrail_logic(messages: list) -> None:
    """Output guardrail that applies multiple filters with different behaviors."""
    if not messages:
        return

    # Find the last assistant message text
    response_text = None
    for message in reversed(messages):
        if message.get("role") == "assistant":
            for block in message.get("content", []):
                if "text" in block:
                    response_text = block["text"]
                    break
            break

    if not response_text:
        return

    # Phase 1: Check BLOCK-severity filters first
    block_filters = [output_keyword_filter, format_filter]
    violation = run_filters(response_text, block_filters)

    if violation is not None:
        logger.warning(
            f"[COMBINED OUTPUT GUARDRAIL] Blocked response. "
            f"Filter: {violation.filter_name}, Reason: {violation.message}"
        )
        _replace_assistant_response(messages, BLOCKED_RESPONSE_FALLBACK)
        return

    # Phase 2: Apply REDACT-severity filters (PII redaction)
    redact_result = pii_redaction_filter.evaluate(response_text)

    if not redact_result.passed:
        logger.info(
            f"[COMBINED OUTPUT GUARDRAIL] Redacted content from response. "
            f"Filter: {redact_result.filter_name}, Reason: {redact_result.message}"
        )
        _redact_assistant_response(messages, redact_result.redacted_text)
    else:
        logger.debug("[COMBINED OUTPUT GUARDRAIL] Response passed all output filters.")

## Testing the Output Guardrails

We test output guardrails using mock messages that simulate the conversation state after model inference.

In [ ]:
# Test BLOCK behavior
print("Test: BLOCK — Prohibited keyword in output")
mock_messages = [{"role": "assistant", "content": [{"text": "Here is the confidential internal document you requested..."}]}]
output_guardrail_logic(mock_messages)
print(f"  Result: '{mock_messages[0]['content'][0]['text'][:60]}...'")
assert "cannot provide" in mock_messages[0]["content"][0]["text"]

# Test REDACT behavior
print("\nTest: REDACT — PII redaction in output")
mock_messages = [{"role": "assistant", "content": [{"text": "The user's email is john.doe@example.com and phone is 555-123-4567."}]}]
pii_output_guardrail_logic(mock_messages)
redacted = mock_messages[0]["content"][0]["text"]
print(f"  Result: '{redacted}'")
assert "[REDACTED]" in redacted
assert "john.doe@example.com" not in redacted

# Test clean content passes
print("\nTest: Clean content passes through")
mock_messages = [{"role": "assistant", "content": [{"text": "The capital of France is Paris."}]}]
output_guardrail_logic(mock_messages)
print(f"  Result: Message unchanged (passed)")
assert mock_messages[0]["content"][0]["text"] == "The capital of France is Paris."

# Test combined: BLOCK takes priority
print("\nTest: Combined — BLOCK takes priority over REDACT")
mock_messages = [{"role": "assistant", "content": [{"text": "Run this command: sudo rm -rf /tmp/cache to fix the issue."}]}]
combined_output_guardrail_logic(mock_messages)
print(f"  Result: '{mock_messages[0]['content'][0]['text'][:60]}...'")
assert "cannot provide" in mock_messages[0]["content"][0]["text"]

# Test combined: REDACT when no BLOCK violation
print("\nTest: Combined — REDACT when no BLOCK violation")
mock_messages = [{"role": "assistant", "content": [{"text": "Please contact support at help@company.com for assistance."}]}]
combined_output_guardrail_logic(mock_messages)
redacted = mock_messages[0]["content"][0]["text"]
print(f"  Result: '{redacted}'")
assert "[REDACTED]" in redacted
assert "help@company.com" not in redacted

## HookProvider: Wrapping Output Guardrails for Agent Registration

In strands-agents 1.40.0, hooks are registered via `HookProvider` classes. The `AfterInvocationEvent` gives access to `event.agent.messages` which contains the full conversation including the assistant's response.

In [ ]:
class OutputGuardrailHook(HookProvider):
    """HookProvider that applies the combined output guardrail after each invocation."""

    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(AfterInvocationEvent, self._validate_output)

    def _validate_output(self, event: AfterInvocationEvent) -> None:
        messages = event.agent.messages
        combined_output_guardrail_logic(messages)


print("OutputGuardrailHook defined successfully.")
print("Register with: Agent(hooks=[OutputGuardrailHook()])")

## Attaching to a Live Agent

Register output guardrails using the `hooks` parameter with `HookProvider` instances:

In [ ]:
try:
    from strands import Agent
    from strands.models.bedrock import BedrockModel

    model = BedrockModel(model_id="us.anthropic.claude-sonnet-4-5-20250929-v1:0")

    agent = Agent(
        model=model,
        system_prompt="You are a helpful assistant.",
        hooks=[OutputGuardrailHook()],
    )

    print("Agent created with output guardrail attached.")
    print("Testing with a safe request...")
    response = agent("What is the capital of France?")
    print(f"  Response: {response}")

    print("\nTesting with a request that might produce PII...")
    response = agent("Generate a fake contact card with name, email, and phone number.")
    print(f"  Response: {response}")

except Exception as e:
    print(f"Skipping live agent demo: {e}")
    print("(This is expected if no model provider is configured)")

## Summary

In this notebook you learned:
1. **BLOCK behavior** — replace the entire response when prohibited content is detected
2. **REDACT behavior** — replace only matched patterns (PII) while preserving the rest
3. How to compose multiple output filters with mixed severity levels
4. How to test output guardrails with mock messages
5. How to wrap guardrail logic in a `HookProvider` for agent registration
6. How to attach output guardrails to a live agent

**Next Steps:** See `03_content_filters.ipynb` to learn how to build custom content filter classes.